In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("ClimateEDA")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark sürümü: {spark.version}")
print("Spark Session hazır ✅")

Spark sürümü: 3.5.0
Spark Session hazır ✅


In [2]:
# Silver tablosunu oku
SILVER_PATH = "/home/jovyan/work/delta-lake/silver/climate_events"

df = spark.read.format("delta").load(SILVER_PATH)

print(f"Toplam satır sayısı: {df.count():,}")
print(f"Toplam sütun sayısı: {len(df.columns)}")
print("\nŞema:")
df.printSchema()

Toplam satır sayısı: 5,000
Toplam sütun sayısı: 22

Şema:
root
 |-- event_type: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- measurement_date: date (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- snow_depth_mm: double (nullable = true)
 |-- avg_wind_dir_deg: double (nullable = true)
 |-- avg_wind_speed_kmh: double (nullable = true)
 |-- peak_wind_gust_kmh: double (nullable = true)
 |-- avg_sea_level_pres_hpa: double (nullable = true)
 |-- sunshine_total_min: double (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = tru

In [5]:
import pandas as pd

print("=== TEMEL İSTATİSTİKLER ===\n")

numeric_cols = ["avg_temp_c", "min_temp_c", "max_temp_c", "precipitation_mm", "snow_depth_mm"]

stats = df.select(numeric_cols).describe().toPandas()
stats = stats.set_index("summary")
print(stats.to_string())

=== TEMEL İSTATİSTİKLER ===

                 avg_temp_c          min_temp_c          max_temp_c    precipitation_mm       snow_depth_mm
summary                                                                                                    
count                  5000                3719                4072                3195                   3
mean     18.086119999999994  11.214869588599086  25.258423379174854  1.6586854460093883               110.0
stddev    9.243158688571157   8.351186129315648  10.408331544072867   6.426648069545371  114.59057552870567
min                    -2.1               -11.0                 1.3                 0.0                20.0
max                    37.8                28.9                48.0               130.3               239.0


avg_temp_c tamamen dolu (5000/5000), ortalama 18°C, -2.1 ile 37.8 arası — tek bir şehrin verisi bu

min_temp_c ve max_temp_c eksik değerler var (3719 ve 4072 dolu)

snow_depth_mm neredeyse tamamen boş (sadece 3 kayıt)

precipitation_mm 3195 dolu

In [8]:
import builtins

print("=== EKSİK DEĞER ANALİZİ ===\n")

total = df.count()

null_counts = []
for col_name in df.columns:
    null_count = df.filter(col(col_name).isNull()).count()
    null_pct = (null_count / total) * 100
    null_counts.append((col_name, null_count, builtins.round(null_pct, 2)))

null_df = pd.DataFrame(null_counts, columns=["sütun", "eksik_sayı", "eksik_yüzde"])
null_df = null_df.sort_values("eksik_yüzde", ascending=False)
print(null_df.to_string(index=False))

=== EKSİK DEĞER ANALİZİ ===

                 sütun  eksik_sayı  eksik_yüzde
    avg_wind_speed_kmh        5000       100.00
      avg_wind_dir_deg        5000       100.00
    peak_wind_gust_kmh        5000       100.00
avg_sea_level_pres_hpa        5000       100.00
    sunshine_total_min        5000       100.00
         snow_depth_mm        4997        99.94
      precipitation_mm        1805        36.10
            min_temp_c        1281        25.62
            max_temp_c         928        18.56
       kafka_timestamp           0         0.00
                   day           0         0.00
                 month           0         0.00
                  year           0         0.00
       event_timestamp           0         0.00
        ingestion_time           0         0.00
            event_type           0         0.00
            station_id           0         0.00
            avg_temp_c           0         0.00
                season           0         0.00
      measu

In [9]:
print("=== ZAMAN SERİSİ ANALİZİ ===\n")

# Yıllık ortalama sıcaklık trendi
yearly = df.groupBy("year").agg(
    avg("avg_temp_c").alias("ort_sicaklik"),
    count("*").alias("kayit_sayisi")
).orderBy("year").toPandas()

print("Yıllık Ortalama Sıcaklık:")
print(yearly.to_string(index=False))

=== ZAMAN SERİSİ ANALİZİ ===

Yıllık Ortalama Sıcaklık:
 year  ort_sicaklik  kayit_sayisi
 1957     18.167391           184
 1958     16.907945           365
 1961     20.255000            60
 1962     15.402247            89
 1973      8.753846            13
 1974     25.066667             6
 1975     13.523077            26
 1976     18.579104            67
 1977     21.360976            41
 1978     14.955952            84
 1979     18.610274           146
 1980     17.711979           192
 1981     16.890303           165
 1982     15.062366            93
 1983     15.216822           107
 1984     17.963380            71
 1985     14.502500            40
 1986     15.965517            29
 1987     18.642105            95
 1988     15.559740            77
 1989     17.577692           130
 1990      6.940000             5
 1991     11.916667            12
 1992     17.148387            31
 1993     18.204762            42
 1994     17.112500            24
 1995     13.720690       

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("İklim Verisi EDA", fontsize=16, fontweight='bold')

# 1. Yıllık sıcaklık trendi
axes[0,0].plot(yearly["year"], yearly["ort_sicaklik"], marker='o', markersize=3, linewidth=1.5, color='tomato')
axes[0,0].set_title("Yıllık Ortalama Sıcaklık Trendi")
axes[0,0].set_xlabel("Yıl")
axes[0,0].set_ylabel("Sıcaklık (°C)")
axes[0,0].grid(True, alpha=0.3)

# 2. Aylık ortalama sıcaklık
monthly = df.groupBy("month").agg(avg("avg_temp_c").alias("ort_sicaklik")).orderBy("month").toPandas()
ay_isimleri = ["Oca","Şub","Mar","Nis","May","Haz","Tem","Ağu","Eyl","Eki","Kas","Ara"]
axes[0,1].bar(monthly["month"], monthly["ort_sicaklik"], color='steelblue', alpha=0.8)
axes[0,1].set_title("Aylık Ortalama Sıcaklık")
axes[0,1].set_xlabel("Ay")
axes[0,1].set_ylabel("Sıcaklık (°C)")
axes[0,1].set_xticks(range(1,13))
axes[0,1].set_xticklabels(ay_isimleri)
axes[0,1].grid(True, alpha=0.3, axis='y')

# 3. Sıcaklık dağılımı
temp_data = df.select("avg_temp_c").dropna().toPandas()
axes[1,0].hist(temp_data["avg_temp_c"], bins=40, color='mediumseagreen', alpha=0.8, edgecolor='white')
axes[1,0].set_title("Sıcaklık Dağılımı (Histogram)")
axes[1,0].set_xlabel("Sıcaklık (°C)")
axes[1,0].set_ylabel("Frekans")
axes[1,0].grid(True, alpha=0.3, axis='y')

# 4. Sezon dağılımı
season_counts = df.groupBy("season").count().toPandas()
axes[1,1].pie(season_counts["count"], labels=season_counts["season"], autopct='%1.1f%%',
              colors=['#FF9999','#66B2FF','#99FF99','#FFCC99'])
axes[1,1].set_title("Sezon Dağılımı")

plt.tight_layout()
plt.savefig("/home/jovyan/work/notebooks/eda_gorseller.png", dpi=150, bbox_inches='tight')
plt.show()
print("Görseller kaydedildi ✅")

Görseller kaydedildi ✅


In [11]:
print("=== YAĞIŞ VE SEZON ANALİZİ ===\n")

# Sezon bazında istatistikler
season_stats = df.groupBy("season").agg(
    count("*").alias("kayit_sayisi"),
    avg("avg_temp_c").alias("ort_sicaklik"),
    avg("precipitation_mm").alias("ort_yagis")
).orderBy("season").toPandas()

print("Sezon Bazında İstatistikler:")
print(season_stats.to_string(index=False))

print("\n")

# Yağış dağılımı
print("Yağış Dağılımı:")
rain_stats = df.select("precipitation_mm").dropna().describe().toPandas().set_index("summary")
print(rain_stats.to_string())

=== YAĞIŞ VE SEZON ANALİZİ ===

Sezon Bazında İstatistikler:
season  kayit_sayisi  ort_sicaklik  ort_yagis
Autumn          1397     18.594775   1.001735
Spring          1210     17.224132   2.900130
Summer          1282     28.699376   0.864074
Winter          1111      6.138524   2.080925


Yağış Dağılımı:
           precipitation_mm
summary                    
count                  3195
mean     1.6586854460093883
stddev    6.426648069545371
min                     0.0
max                   130.3


In [13]:
print("=== EDA ÖZET BULGULAR ===\n")

print("1. VERİ KALİTESİ:")
print("   - avg_temp_c: Tam dolu, analiz için güvenilir ana değişken")
print("   - min/max_temp_c: %18-25 eksik değer var")
print("   - precipitation_mm: %36 eksik")
print("   - snow_depth_mm, wind, pressure, sunshine: Kullanılamaz (>%99 eksik)")

print("\n2. SICAKLIK BULGULARI:")
print(f"   - Ortalama sıcaklık: 18.1°C")
print(f"   - Aralık: -2.1°C ile 37.8°C")
print(f"   - En sıcak ay: Temmuz (~30°C)")
print(f"   - En soğuk ay: Ocak (~5°C)")
print(f"   - Yaz ortalaması: 28.7°C, Kış ortalaması: 6.1°C")

print("\n3. ZAMAN SERİSİ:")
print("   - Veri aralığı: 1957-2010 (53 yıl)")
print("   - 1970-1990 arası veri eksikliği var (az kayıt sayısı)")
print("   - 2003-2010 arası veri en yoğun dönem")

print("\n4. YAĞIŞ BULGULARI:")
print("   - Ortalama günlük yağış: 1.66 mm")
print("   - Maksimum: 130.3 mm (aşırı yağış olayları var)")
print("   - İlkbahar en yağışlı sezon (2.9 mm ort.)")
print("   - Yaz en kurak sezon (0.86 mm ort.)")

print("\n5. SEZON DAĞILIMI:")
print("   - Dengeli dağılım: Sonbahar %27.9, Yaz %25.6, İlkbahar %24.2, Kış %22.2")

=== EDA ÖZET BULGULAR ===

1. VERİ KALİTESİ:
   - avg_temp_c: Tam dolu, analiz için güvenilir ana değişken
   - min/max_temp_c: %18-25 eksik değer var
   - precipitation_mm: %36 eksik
   - snow_depth_mm, wind, pressure, sunshine: Kullanılamaz (>%99 eksik)

2. SICAKLIK BULGULARI:
   - Ortalama sıcaklık: 18.1°C
   - Aralık: -2.1°C ile 37.8°C
   - En sıcak ay: Temmuz (~30°C)
   - En soğuk ay: Ocak (~5°C)
   - Yaz ortalaması: 28.7°C, Kış ortalaması: 6.1°C

3. ZAMAN SERİSİ:
   - Veri aralığı: 1957-2010 (53 yıl)
   - 1970-1990 arası veri eksikliği var (az kayıt sayısı)
   - 2003-2010 arası veri en yoğun dönem

4. YAĞIŞ BULGULARI:
   - Ortalama günlük yağış: 1.66 mm
   - Maksimum: 130.3 mm (aşırı yağış olayları var)
   - İlkbahar en yağışlı sezon (2.9 mm ort.)
   - Yaz en kurak sezon (0.86 mm ort.)

5. SEZON DAĞILIMI:
   - Dengeli dağılım: Sonbahar %27.9, Yaz %25.6, İlkbahar %24.2, Kış %22.2
